In [1]:
import pandas as pd

In [2]:
df_wnir_train = pd.read_parquet(
    "../data/interim/wnir_train.parquet", engine="pyarrow"
).drop(
    columns=[
        "longitude",
        "latitude",
        "area",
        "room_count",
        "floor",
        "build_year",
        "balcony",
        "administrative_district",
        "date",
        "year",
        "month",
        "day",
        "price_normalized",
    ]
)
df_clusters_train = pd.read_parquet(
    "../data/interim/clusters_train.parquet", engine="pyarrow"
).drop(columns=["date", "year", "month", "day", "price_normalized"])

df_wnir_valid = pd.read_parquet(
    "../data/interim/wnir_valid.parquet", engine="pyarrow"
).drop(
    columns=[
        "longitude",
        "latitude",
        "area",
        "room_count",
        "floor",
        "build_year",
        "balcony",
        "administrative_district",
        "date",
        "year",
        "month",
        "day",
        "price_normalized",
    ]
)
df_clusters_valid = pd.read_parquet(
    "../data/interim/clusters_valid.parquet", engine="pyarrow"
).drop(columns=["date", "year", "month", "day", "price_normalized"])

df_wnir_test = pd.read_parquet(
    "../data/interim/wnir_test.parquet", engine="pyarrow"
).drop(
    columns=[
        "longitude",
        "latitude",
        "area",
        "room_count",
        "floor",
        "build_year",
        "balcony",
        "administrative_district",
        "date",
        "year",
        "month",
        "day",
        "price_normalized",
    ]
)
df_clusters_test = pd.read_parquet(
    "../data/interim/clusters_test.parquet", engine="pyarrow"
).drop(columns=["date", "year", "month", "day", "price_normalized"])

In [ ]:
df_clusters_train[df_clusters_train["market_type"] == "primary"].drop(
    columns=["market_type"]
)mns=['market_type'])
df_clusters_valid = df_clusters_valid[df_clusters_valid['market_type'] == 'primary'].drop(columns=['market_type'])
df_clusters_test = df_clusters_test[df_clusters_test['market_type'] == 'primary'].drop(columns=['market_type'])

In [4]:
df_wnir_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 767389 entries, 0 to 767388
Data columns (total 41 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   price_per_square_meter_normalized  767389 non-null  float32
 1   wnir_p_value_100                   767389 non-null  float32
 2   wnir_s_value_100                   767389 non-null  float32
 3   wnir_s_mean_100                    767389 non-null  float32
 4   wnir_s_std_100                     767389 non-null  float32
 5   wnir_s_min_100                     767389 non-null  float32
 6   wnir_s_max_100                     767389 non-null  float32
 7   wnir_s_median_100                  767389 non-null  float32
 8   wnir_s_count_100                   767389 non-null  float32
 9   wnir_p_value_500                   767389 non-null  float32
 10  wnir_s_value_500                   767389 non-null  float32
 11  wnir_s_mean_500                    767389 non-null

In [5]:
df_clusters_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 767389 entries, 0 to 767388
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   price_per_square_meter_normalized  767389 non-null  float32
 1   umap_1                             767389 non-null  float32
 2   umap_2                             767389 non-null  float32
 3   umap_3                             767389 non-null  float32
 4   umap_4                             767389 non-null  float32
 5   umap_5                             767389 non-null  float32
 6   umap_6                             767389 non-null  float32
 7   umap_7                             767389 non-null  float32
 8   umap_8                             767389 non-null  float32
 9   umap_9                             767389 non-null  float32
 10  umap_10                            767389 non-null  float32
 11  cluster                            767389 non-null

In [6]:
# Используем how='inner', чтобы оставить только пересечение по индексам.
# Теперь размер df_merged_train будет ровно 767 389 строк и никаких пустых значений.

df_merged_train = df_clusters_train.join(
    df_wnir_train.drop(
        columns=df_clusters_train.columns.intersection(df_wnir_train.columns)
    ),
    how="inner",
)

df_merged_valid = df_clusters_valid.join(
    df_wnir_valid.drop(
        columns=df_clusters_valid.columns.intersection(df_wnir_valid.columns)
    ),
    how="inner",
)

df_merged_test = df_clusters_test.join(
    df_wnir_test.drop(
        columns=df_clusters_test.columns.intersection(df_wnir_test.columns)
    ),
    how="inner",
)

print(f"Размер train после inner join: {df_merged_train.shape}")

Размер train после inner join: (767389, 52)


In [7]:
df_merged_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 767389 entries, 0 to 767388
Data columns (total 52 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   price_per_square_meter_normalized  767389 non-null  float32
 1   umap_1                             767389 non-null  float32
 2   umap_2                             767389 non-null  float32
 3   umap_3                             767389 non-null  float32
 4   umap_4                             767389 non-null  float32
 5   umap_5                             767389 non-null  float32
 6   umap_6                             767389 non-null  float32
 7   umap_7                             767389 non-null  float32
 8   umap_8                             767389 non-null  float32
 9   umap_9                             767389 non-null  float32
 10  umap_10                            767389 non-null  float32
 11  cluster                            767389 non-null

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

# ===========================
# 1. Предсказываем price_per_square_meter_normalized
# ===========================

target1 = "price_per_square_meter_normalized"
exclude1 = [
    target1,
    "wnir_p_value_100",
    "wnir_p_value_100",
    "wnir_p_value_500",
    "wnir_p_value_1000",
    "wnir_p_value_5000",
    "wnir_p_value_10000",
    "cluster",
]

X1 = df_merged_train.drop(columns=exclude1)
y1 = df_merged_train[target1]

# Train-test split
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)

# Pipeline
pipeline1 = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LinearRegression()),
    ]
)

pipeline1.fit(X1_train, y1_train)
y1_pred = pipeline1.predict(X1_test)

# Метрики
print("=== МОДЕЛЬ 1: Предсказание price_per_square_meter_normalized ===")
print(f"R² score:     {r2_score(y1_test, y1_pred):.5f}")
print(f"MAE:          {mean_absolute_error(y1_test, y1_pred):.5f}")
print(f"RMSE:         {np.sqrt(mean_squared_error(y1_test, y1_pred)):.5f}")
print(f"MAPE:         {(np.abs((y1_test - y1_pred) / y1_test).mean() * 100):.2f}%")
print("-" * 60)

# Feature Importance (абсолютное значение коэффициентов)
model1 = pipeline1.named_steps["lr"]
feature_names1 = X1.columns
importances1 = pd.DataFrame(
    {"feature": feature_names1, "importance": np.abs(model1.coef_)}
).sort_values("importance", ascending=False)

print("ТОП-15 самых важных признаков:")
print(importances1.head(15))

=== МОДЕЛЬ 1: Предсказание price_per_square_meter_normalized ===
R² score:     0.22954
MAE:          74958.21875
RMSE:         129943.09154
MAPE:         28.25%
------------------------------------------------------------
ТОП-15 самых важных признаков:
               feature    importance
5               umap_6  48694.023438
2               umap_3  38240.582031
4               umap_5  29443.505859
6               umap_7  20017.314453
0               umap_1  19355.427734
3               umap_4  15445.485352
8               umap_9  11645.125000
1               umap_2   7108.419922
35     wnir_s_max_5000   3903.791748
36  wnir_s_median_5000   3430.337646
7               umap_8   2698.868652
44  wnir_s_count_10000   1782.586670
33     wnir_s_std_5000   1726.948120
9              umap_10   1480.955322
32    wnir_s_mean_5000   1477.098633


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")


def train_lr_per_cluster(df, target_col, exclude_cols):
    print(f"\n{'=' * 70}")
    print(f"ОБУЧЕНИЕ МОДЕЛЕЙ ДЛЯ ПРЕДСКАЗАНИЯ: {target_col.upper()}")
    print(f"{'=' * 70}\n")

    # Получаем список уникальных кластеров и сортируем их
    unique_clusters = sorted(df["cluster"].unique())

    # Собираем метрики по всем кластерам для итогового сравнения (опционально)
    cluster_metrics = []

    for cluster_id in unique_clusters:
        # 1. Фильтруем данные только для текущего кластера
        df_cluster = df[df["cluster"] == cluster_id].copy()

        # Если в кластере слишком мало данных (например, меньше 10), пропускаем
        if len(df_cluster) < 10:
            print(
                f"--- Кластер {cluster_id}: пропущен (слишком мало данных: {len(df_cluster)} строк) ---"
            )
            continue

        print(f"\n>>> КЛАСТЕР {cluster_id} (Размер выборки: {len(df_cluster)} строк)")

        # 2. Формируем X и y
        # Обязательно исключаем саму колонку 'cluster', так как здесь она константа
        cols_to_drop = list(set(exclude_cols + ["cluster"]))

        X = df_cluster.drop(columns=cols_to_drop, errors="ignore")
        y = df_cluster[target_col]

        # 3. Разбиваем на train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # 4. Создаем и обучаем Pipeline
        pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("lr", LinearRegression()),
            ]
        )

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        # 5. Считаем метрики
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mape = mean_absolute_percentage_error(y_test, y_pred) * 100

        print(f"Метрики:")
        print(
            f"R² score: {r2:.5f} | MAE: {mae:.5f} | RMSE: {rmse:.5f} | MAPE: {mape:.2f}%"
        )

        # 6. Достаем Feature Importance
        # Находим шаг 'lr' в пайплайне и берем веса
        model = pipeline.named_steps["lr"]
        feature_names = X.columns

        importances = pd.DataFrame(
            {
                "feature": feature_names,
                "weight": model.coef_,
                "importance_abs": np.abs(model.coef_),
            }
        ).sort_values("importance_abs", ascending=False)

        print("ТОП-5 самых важных признаков:")
        # Выводим топ-5, чтобы не засорять консоль (можно поменять на 10 или 15)
        print(importances.head(5)[["feature", "weight"]].to_string(index=False))
        print("-" * 50)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_percentage_error,
)  # <-- Добавлен MAPE
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

# 1. Задаем таргеты и колонки для исключения
all_targets = [
    "price_per_square_meter_normalized",
    "wnir_p_value_100",
    "wnir_p_value_500",
    "wnir_p_value_1000",
    "wnir_p_value_5000",
    "wnir_p_value_10000",
]

# Исключаем все таргеты и колонку cluster (чтобы глобальная модель не читерила)
exclude_cols = all_targets.copy() + ["cluster"]

results = []

print("Обучение ГЛОБАЛЬНЫХ моделей и моделей ПО КЛАСТЕРАМ. Пожалуйста, подождите...\n")

# ==========================================
# ШАГ 1: ОБУЧАЕМ ГЛОБАЛЬНЫЕ МОДЕЛИ (НА ВСЕХ ДАННЫХ)
# ==========================================
for target in all_targets:
    X_global = df_merged_train.drop(columns=exclude_cols, errors="ignore")
    y_global = df_merged_train[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X_global, y_global, test_size=0.2, random_state=42
    )

    pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ]
    )

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    # Считаем обе метрики
    r2 = r2_score(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds)

    # Сокращаем имена для красоты таблицы
    short_target = target.replace("price_per_square_meter_normalized", "PRICE").replace(
        "wnir_p_value_", "P_VAL_"
    )

    results.append(
        {
            "Model_Type": "0_GLOBAL (Все данные)",
            "Target": short_target,
            "R2_Score": r2,
            "MAPE": mape,  # <-- Сохраняем MAPE
        }
    )

# ==========================================
# ШАГ 2: ОБУЧАЕМ ЛОКАЛЬНЫЕ МОДЕЛИ (ПО КЛАСТЕРАМ)
# ==========================================
unique_clusters = sorted(df_merged_train["cluster"].unique())

for target in all_targets:
    short_target = target.replace("price_per_square_meter_normalized", "PRICE").replace(
        "wnir_p_value_", "P_VAL_"
    )

    for cluster_id in unique_clusters:
        df_cluster = df_merged_train[df_merged_train["cluster"] == cluster_id]

        if len(df_cluster) < 20:
            continue

        X_clust = df_cluster.drop(columns=exclude_cols, errors="ignore")
        y_clust = df_cluster[target]

        X_train, X_test, y_train, y_test = train_test_split(
            X_clust, y_clust, test_size=0.2, random_state=42
        )

        pipe = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("lr", LinearRegression()),
            ]
        )

        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        # Считаем обе метрики
        r2 = r2_score(y_test, preds)
        mape = mean_absolute_percentage_error(y_test, preds)

        results.append(
            {
                "Model_Type": f"Cluster_{cluster_id}",
                "Target": short_target,
                "R2_Score": r2,
                "MAPE": mape,  # <-- Сохраняем MAPE
            }
        )

# ==========================================
# ШАГ 3: ФОРМИРУЕМ И ВЫВОДИМ СВОДНУЮ ТАБЛИЦУ
# ==========================================
df_results = pd.DataFrame(results)

# Создаем отдельные сводные таблицы для R2 и MAPE
pivot_r2 = df_results.pivot(index="Model_Type", columns="Target", values="R2_Score")
pivot_mape = df_results.pivot(index="Model_Type", columns="Target", values="MAPE")

# Добавляем суффиксы к названиям колонок, чтобы не запутаться
pivot_r2.columns = [f"{col} (R2)" for col in pivot_r2.columns]
pivot_mape.columns = [f"{col} (MAPE)" for col in pivot_mape.columns]

# Склеиваем таблицы по столбцам
pivot_results = pd.concat([pivot_r2, pivot_mape], axis=1)

# Считаем средние значения по строке
pivot_results["MEAN_R2_ALL"] = pivot_r2.mean(axis=1)
pivot_results["MEAN_MAPE_ALL"] = pivot_mape.mean(axis=1)

# Упорядочиваем колонки, чтобы R2 и MAPE одного таргета стояли рядом
ordered_cols = []
for target in df_results["Target"].unique():
    ordered_cols.append(f"{target} (R2)")
    ordered_cols.append(f"{target} (MAPE)")
ordered_cols.extend(["MEAN_R2_ALL", "MEAN_MAPE_ALL"])

pivot_results = pivot_results[ordered_cols]

# Отделяем Глобальную модель от кластерных
global_row = pivot_results[pivot_results.index.str.startswith("0_GLOBAL")]
cluster_rows = pivot_results[
    ~pivot_results.index.str.startswith("0_GLOBAL")
].sort_values(by="MEAN_R2_ALL", ascending=False)

# Склеиваем обратно
final_table = pd.concat([global_row, cluster_rows])
final_table.index = final_table.index.str.replace("0_GLOBAL", "GLOBAL")

print("=== СРАВНЕНИЕ ГЛОБАЛЬНОЙ МОДЕЛИ И КЛАСТЕРОВ (R2 и MAPE) ===")
print("Для R2: Зеленый цвет - хорошо (ближе к 1), Красный - плохо.")
print(
    "Для MAPE: Зеленый цвет - хорошо (меньше ошибка), Красный - плохо (высокая ошибка).\n"
)

# Разделяем колонки для разной цветовой раскраски
r2_columns = [col for col in final_table.columns if "R2" in col]
mape_columns = [col for col in final_table.columns if "MAPE" in col]

# Применяем раскраску: для R2 обычная палитра (зеленый сверху), для MAPE - инвертированная (зеленый снизу)
styled_table = (
    final_table.style.background_gradient(
        cmap="RdYlGn", vmin=0, vmax=1, subset=r2_columns
    )
    .background_gradient(
        cmap="RdYlGn_r", vmin=0, subset=mape_columns
    )  # _r инвертирует цвета
    .format("{:.4f}")
)

display(styled_table)

Обучение ГЛОБАЛЬНЫХ моделей и моделей ПО КЛАСТЕРАМ. Пожалуйста, подождите...

=== СРАВНЕНИЕ ГЛОБАЛЬНОЙ МОДЕЛИ И КЛАСТЕРОВ (R2 и MAPE) ===
Для R2: Зеленый цвет - хорошо (ближе к 1), Красный - плохо.
Для MAPE: Зеленый цвет - хорошо (меньше ошибка), Красный - плохо (высокая ошибка).



,PRICE (R2),PRICE (MAPE),P_VAL_100 (R2),P_VAL_100 (MAPE),P_VAL_500 (R2),P_VAL_500 (MAPE),P_VAL_1000 (R2),P_VAL_1000 (MAPE),P_VAL_5000 (R2),P_VAL_5000 (MAPE),P_VAL_10000 (R2),P_VAL_10000 (MAPE),MEAN_R2_ALL,MEAN_MAPE_ALL
Model_Type,,,,,,,,,,,,,,
GLOBAL (Все данные),0.2295,0.2825,0.1972,0.3068,0.1972,0.3068,0.1972,0.3068,0.1980,0.3070,0.1980,0.3070,0.2028,0.3028
Cluster_104,0.5143,0.0864,0.5796,0.2671,0.5796,0.2671,0.5796,0.2671,0.5796,0.2671,0.5796,0.2671,0.5687,0.2370
Cluster_85,0.7425,0.1600,0.4050,0.2822,0.4051,0.2820,0.4051,0.2820,0.4051,0.2819,0.4051,0.2819,0.4613,0.2617
Cluster_98,0.5291,0.1254,0.3980,0.2909,0.3980,0.2909,0.3980,0.2909,0.3980,0.2910,0.3980,0.2910,0.4199,0.2633
Cluster_34,0.4536,0.1281,0.4121,0.2882,0.4121,0.2882,0.4121,0.2882,0.4120,0.2891,0.4120,0.2891,0.4190,0.2618
Cluster_37,0.4101,0.1649,0.4023,0.2719,0.4023,0.2718,0.4023,0.2718,0.4019,0.2718,0.4019,0.2718,0.4034,0.2540
Cluster_157,0.3573,0.1198,0.4061,0.2867,0.4061,0.2867,0.4061,0.2867,0.4061,0.2866,0.4061,0.2866,0.3980,0.2589
Cluster_53,0.3887,0.1548,0.3957,0.2629,0.3957,0.2628,0.3957,0.2628,0.3951,0.2636,0.3951,0.2636,0.3943,0.2451
Cluster_95,0.2510,0.2026,0.4161,0.2958,0.4161,0.2958,0.4162,0.2958,0.4164,0.2955,0.4164,0.2955,0.3887,0.2802
